# JS09 — Lab 4: ANNOY vs. FAISS vs. HNSW

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** membandingkan waktu **build** dan **query** ketiga library ANN pada 1.000.000 vektor 5D, lalu mengukur recall masing-masing terhadap hasil exact FAISS.

**Prasyarat:** library `annoy`, `faiss`, dan `hnswlib` (dipasang otomatis bila belum ada). Tidak ada dataset eksternal.

> Jalankan di **Google Colab**: `Runtime` ▸ `Run all`. Notebook ini cukup memakai CPU, tidak perlu GPU.


# Lab 4 — ANNOY vs. FAISS vs. HNSW

Modul: `JS09 › Lab 4 — ANNOY vs. FAISS vs. HNSW`

Percobaan ini memakai data yang jauh lebih besar: **1.000.000 vektor 5D**, query acak, `k = 10`. Ketiga library diukur waktu *build* dan waktu *query*-nya pada data yang sama:

- **ANNOY** — 10 tree
- **FAISS** — `IndexFlatL2` (exact brute force teroptimasi)
- **HNSW** — `hnswlib`, `M = 16`, `ef_construction = 200`, `ef = 50`

> **Perkiraan waktu di Colab CPU:** ANNOY ±20–30 detik (karena `add_item` dipanggil satu per satu dari Python), FAISS Flat ±0,1 detik, HNSW ±2–3 menit untuk membangun graf 1 juta node. Modul menyarankan menurunkan `n_data` ke `100_000` bila RAM/waktu terbatas — ubah baris `n_data = 1_000_000` pada cell berikut.

> **RAM:** 1.000.000 × 5 float32 ≈ 20 MB, jadi aman. Yang lama adalah proses build HNSW, bukan memorinya.

In [ ]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# Build 1mio rows with 5D
# ===============================
n_data = 1_000_000   # try 100_000 if you have limited memory
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

**Contoh output modul** (angka pasti berbeda di tiap mesin):

```text
=== Annoy ===
Build time: 20.291711807250977 detik
Query time: 0.00025844573974609375 detik
Neighbors: [652544, 105406, 139716, 722254, 898386] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.017807483673095703 detik
Query time: 0.012749195098876953 detik
Neighbors: [652544 105406 139716 722254 898386] ...

=== HNSW (hnswlib) ===
Build time: 156.97257900238037 detik
Query time: 0.00021314620971679688 detik
Neighbors: [652544 105406 139716 722254 898386] ...
```

**Tabel untuk laporan** — isi dari output di atas:

| Algoritma | Build time (detik) | Query time (detik) | Catatan |
|---|---|---|---|
| ANNOY (10 tree) | TBD | TBD | build lambat karena `add_item` per vektor dari Python |
| FAISS `IndexFlatL2` | TBD | TBD | exact, build paling cepat |
| HNSW (`M=16`, `ef=50`) | TBD | TBD | build paling lama, query paling cepat |

Catatan: variabel `build_time` dan `query_time` ditimpa di setiap blok kode modul, jadi angka ANNOY dan FAISS hilang setelah blok HNSW berjalan — catat dari output cetak, bukan dari variabel.

### [tambahan] Ukur ulang dengan 100 query + recall@10

Waktu satu query sangat bising. Cell ini memakai 100 query acak, menjalankan ketiga indeks yang **sudah dibangun** di atas, dan menghitung recall@10 terhadap hasil exact `IndexFlatL2`.

In [ ]:
# [tambahan] Rekap query time 100 titik + recall@10 vs FAISS Flat
import pandas as pd

rng = np.random.default_rng(0)
qidx = rng.choice(n_data, size=100, replace=False)
Q = X[qidx]

start = time.time()
_, gt = faiss_index.search(Q, k)
t_flat = (time.time() - start) * 1000 / len(Q)

start = time.time()
pred_ann = [ann_index.get_nns_by_vector(q, k) for q in Q]
t_ann = (time.time() - start) * 1000 / len(Q)

start = time.time()
pred_hnsw, _ = hnsw_index.knn_query(Q, k=k)
t_hnsw = (time.time() - start) * 1000 / len(Q)

def recall_at_k(pred, gt, k=10):
    return float(np.mean([len(set(np.asarray(p)[:k]) & set(np.asarray(g)[:k])) / k
                          for p, g in zip(pred, gt)]))

rekap_lab4 = pd.DataFrame([
    {'algoritma': 'FAISS Flat (exact)',  'query (ms/query)': t_flat, 'recall@10': 1.0},
    {'algoritma': 'ANNOY (10 tree)',     'query (ms/query)': t_ann,  'recall@10': recall_at_k(pred_ann, gt)},
    {'algoritma': 'HNSW (M=16, ef=50)', 'query (ms/query)': t_hnsw, 'recall@10': recall_at_k(pred_hnsw, gt)},
]).round(4)
rekap_lab4

**Analisis Lab 4:** TBD

*(Jelaskan: (1) mengapa build ANNOY jauh lebih lama dari FAISS tetapi lebih cepat dari HNSW, (2) mengapa query FAISS Flat paling lambat padahal build-nya paling cepat, (3) mengapa HNSW hampir menyamai exact pada recall tetapi jauh lebih cepat, (4) kapan Anda memilih masing-masing library.)*

---
## Kesimpulan Lab 4

**Urutan build time (cepat → lambat):** TBD

**Urutan query time (cepat → lambat):** TBD

**Recall@10 tiap algoritma:** TBD

**Rekomendasi pemakaian untuk data 1 juta vektor:** TBD